# P21 EuroSAT: Colab / Kaggle setup

This notebook prepares a GPU runtime for training:

1. It detects whether it runs on Colab or Kaggle, and checks the GPU.
2. It clones the repository and installs it, keeping the platform's own CUDA build of `torch`.
3. It downloads EuroSAT (RGB and 13-band), verifies the committed splits, and packs the images into `data/cache/`.
4. It runs the unit tests and loads one batch of each input type on the GPU.

**Colab:** choose *Runtime → Change runtime type → T4 GPU* before running.
**Kaggle:** turn on *Settings → Accelerator → GPU T4* and *Settings → Internet → On*.

The first run takes about 10–15 minutes, mostly for the 2.1 GB 13-band download.

In [ ]:
import os, sys, subprocess
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.exists("/kaggle/working")
BASE = "/content" if ON_COLAB else "/kaggle/working" if ON_KAGGLE else os.getcwd()
print(f"Colab: {ON_COLAB}  Kaggle: {ON_KAGGLE}  base dir: {BASE}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and install
**Option A, GitHub:** set `REPO_URL`. For a private repo, use a token stored in Colab *Secrets* or Kaggle *Add-ons → Secrets*. Never paste the token into the notebook.
**Option B, zip upload (no GitHub yet):** on your laptop run `bash scripts/make_code_zip.sh`, then upload `dist/p21_code.zip`. On Colab, drag it into the *Files* pane (it lands in `/content`). On Kaggle, add it as a dataset. Leave `REPO_URL` unchanged.


In [ ]:
REPO_URL = "https://github.com/<account>/<repo>.git"   # Option A: set after the repo is created
BRANCH = "main"
REPO = f"{BASE}/p21"

if not os.path.exists(REPO):
    if "<account>" not in REPO_URL:
        !git clone -b {BRANCH} {REPO_URL} {REPO}
    else:  # Option B: uploaded zip
        import glob
        zips = glob.glob(f"{BASE}/p21_code.zip") + glob.glob("/kaggle/input/*/p21_code.zip")
        assert zips, "Upload dist/p21_code.zip (see Option B) or set REPO_URL"
        !unzip -q -o {zips[0]} -d {BASE}
%cd {REPO}
!cat COMMIT 2>/dev/null || git log --oneline -1
!pip install -q -r requirements.txt -e .
sys.path.insert(0, f"{REPO}/src")   # make the editable install importable in this kernel


## 2. (Colab, optional) Keep results on Google Drive
Runtimes are wiped when they disconnect. Linking `results/runs` to Drive keeps finished runs, and the training script will skip runs that already exist.

In [ ]:
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RUNS = "/content/drive/MyDrive/p21_runs"
    os.makedirs(DRIVE_RUNS, exist_ok=True)
    os.makedirs("results", exist_ok=True)
    if not os.path.islink("results/runs"):
        !rm -rf results/runs && ln -s {DRIVE_RUNS} results/runs
    !ls -la results/

## 3. Data: download, verify splits, pack the cache

In [ ]:
!python scripts/download_data.py --which all --delete-zip
!python scripts/make_splits.py --check
!python scripts/build_cache.py --which all

## 4. Checks: unit tests and one GPU batch per input type

In [ ]:
!python -m pytest -q

In [ ]:
import time, torch
from eurosat_le.data.loaders import DataConfig, build_loaders

device = "cuda" if torch.cuda.is_available() else "cpu"
for modality in ["rgb", "ms"]:
    loaders = build_loaders(DataConfig(modality=modality, fraction=0.01, num_workers=2, unlabelled=True))
    x, y = next(iter(loaders["train"]))
    xw, xs = next(iter(loaders["unlabelled"]))
    x = x.to(device, non_blocking=True)
    print(f"{modality}: labelled {tuple(x.shape)} on {x.device}, unlabelled weak/strong {tuple(xw.shape)}, "
          f"train/val/test = {len(loaders['train'].dataset)}/{len(loaders['val'].dataset)}/{len(loaders['test'].dataset)}")

## 5. Phase 3 checks: smoke test and baseline
The smoke test checks that every model and input type can memorise one batch (a few minutes on a GPU).
The baseline is ResNet-50 on RGB with 100% of labels. It should reach **≥ 97% test accuracy** (the EuroSAT paper reports about 98.6%). On a T4 it takes about 15–25 minutes.

In [ ]:
!python scripts/smoke_test.py

In [ ]:
!python scripts/train.py --config configs/resnet50.yaml --modality rgb --fraction 1.0 --seed 0

In [ ]:
import json
m = json.load(open("results/runs/resnet50_rgb_100pct_seed0/metrics.json"))
print(f"test accuracy {m['test']['accuracy']:.4f}, macro-F1 {m['test']['macro_f1']:.4f}, best epoch {m['best_epoch']}, "
      f"{m['train_seconds'] / 60:.1f} min")

## 6. Phase 4: tuning sweep and supervised grid
Run sections 1–3 first in every new session. Finished runs are kept on Drive and skipped, so you can re-run a cell after a disconnect.

**Step 1: tuning (9 runs, validation set only).** Afterwards, put the best `lr` per model into `configs/<model>.yaml` on your laptop and commit it, before starting the main grid.

In [ ]:
!python scripts/run_grid.py --plan tune --max-hours 11
!python scripts/collect_results.py

**Step 2: main grid (54 runs).** Use `--dry-run` to see what is done. Split the work with `--models` / `--modalities`, or with `--shard k/n` (for example, Colab runs `--shard 0/2` and Kaggle runs `--shard 1/2`). On Kaggle, keep `--max-hours` under the 12-hour session limit.

In [ ]:
!python scripts/run_grid.py --plan main --dry-run

In [ ]:
!python scripts/run_grid.py --plan main --shard 0/1 --max-hours 11
!python scripts/collect_results.py